In [34]:
import torch

w = torch.rand(5, 5)
m = torch.rand(5, 5)
tril_mask = torch.tril(m, diagonal=-1)
triu_mask = torch.triu(m, diagonal=1)
w_masked = w.masked_fill(tril_mask.bool(), -torch.inf)
s = torch.softmax(w_masked, dim=-1)

print(tril_mask.bool())
print(triu_mask.bool())
print(w_masked)
print(s)

tensor([[False, False, False, False, False],
        [ True, False, False, False, False],
        [ True,  True, False, False, False],
        [ True,  True,  True, False, False],
        [ True,  True,  True,  True, False]])
tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])
tensor([[0.9882, 0.8363, 0.9010, 0.3950, 0.8809],
        [  -inf, 0.5432, 0.2185, 0.3834, 0.3720],
        [  -inf,   -inf, 0.7475, 0.4979, 0.8549],
        [  -inf,   -inf,   -inf, 0.4130, 0.5585],
        [  -inf,   -inf,   -inf,   -inf, 0.3443]])
tensor([[0.2366, 0.2033, 0.2168, 0.1307, 0.2125],
        [0.0000, 0.2926, 0.2115, 0.2494, 0.2465],
        [0.0000, 0.0000, 0.3457, 0.2693, 0.3849],
        [0.0000, 0.0000, 0.0000, 0.4637, 0.5363],
        [0.0000, 0.0000, 0.0000, 0.0000, 1.0000]])


In [35]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.75)
example = torch.ones(6,6)
print(dropout(example))

tensor([[0., 4., 0., 4., 0., 0.],
        [0., 0., 0., 4., 0., 4.],
        [4., 4., 4., 0., 0., 0.],
        [0., 4., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 4.],
        [0., 4., 0., 4., 4., 0.]])


In [36]:
import torch
import torch.nn as nn

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout= nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        num_tokens = x.shape[1]
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1,2)
        attn_scores = torch.masked_fill(attn_scores,
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights = torch.softmax(attn_scores/ keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec


d_in = 6
d_out = 9
context_length = 100
batch = torch.rand((2, 7, 6))
print(batch.shape)
ca = CausalAttention(d_in, d_out, context_length, 0.0)
print(ca(batch).shape)


torch.Size([2, 7, 6])
torch.Size([2, 7, 9])


In [37]:
a = torch.rand((1,2,3,4))

a

tensor([[[[0.7239, 0.3604, 0.1829, 0.2956],
          [0.8646, 0.8010, 0.8044, 0.0733],
          [0.7355, 0.6248, 0.1638, 0.5158]],

         [[0.6000, 0.2299, 0.2890, 0.9078],
          [0.4596, 0.4947, 0.1836, 0.2010],
          [0.9603, 0.6861, 0.4209, 0.8046]]]])

In [38]:
b = a @ a.transpose(2,3)
print(b.shape)

torch.Size([1, 2, 3, 3])


In [39]:
f1 = a[0,0, :, :]
a1 = f1 @ f1.T

print(a1.shape)
print(a1)
b[0][0]


torch.Size([3, 3])
tensor([[0.7748, 1.0834, 0.9401],
        [1.0834, 2.0415, 1.3059],
        [0.9401, 1.3059, 1.2242]])


tensor([[0.7748, 1.0834, 0.9401],
        [1.0834, 2.0415, 1.3059],
        [0.9401, 1.3059, 1.2242]])

In [40]:
f2 = a[0, 1, :, :]
a2 = f2 @ f2.T
print(a2)
b[0][1]

tensor([[1.3205, 0.6250, 1.5860],
        [0.6250, 0.5301, 1.0198],
        [1.5860, 1.0198, 2.2175]])


tensor([[1.3205, 0.6250, 1.5860],
        [0.6250, 0.5301, 1.0198],
        [1.5860, 1.0198, 2.2175]])

In [41]:
a = torch.tensor([[[1,2],[3,4]],[[1,2],[3,4]],[[1, 2],[3, 4]],[[1, 2],[3, 4]]])

a.shape

torch.Size([4, 2, 2])

In [42]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert(d_out % num_heads ==0), "d_out mush be a multiple of num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        # b, num_tokens, d_out
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # b, num_tokens, num_heads, head_dim
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        # b, num_heads, num_tokens, head_dim
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # b, num_heads, num_tokens, num_tokens
        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # b, num_heads, num_tokens, head_dim --> b, num_tokens, num_heads + head_dim=d_out
        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec



In [46]:
class MultiHeadAttentionNew(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        
        self.proj_out = nn.Linear(d_out, d_out)
        self.dropout= nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        # b, num_tokens, d_out
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # b, num_tokens, num_heads, head_dim 
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        # b, num_heads, num_tokens, head_dim
        queries = queries.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        # b, num_heads, num_tokens, num_tokens
        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # b, num_heads, num_tokens, head_dim -> b, num_tokens, num_heads, head_dim
        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.proj_out(context_vec)

        return context_vec



In [47]:
inputs = torch.tensor(
    [[0.43, 0.15, 0.89],
     [0.55, 0.87, 0.66],
     [0.57, 0.85, 0.64],
     [0.22, 0.58, 0.22],
     [0.77, 0.25, 0.10],
     [0.05, 0.80, 0.55]]
)

batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

torch.manual_seed(123)
context_length = batch.shape[1]
d_in, d_out = 3, 6
mha = MultiHeadAttentionNew(
    d_in, d_out, context_length, 0.0, num_heads=3
)
context_vec = mha(batch)

#print(context_vec)
print(context_vec.shape)





torch.Size([2, 6, 3])
torch.Size([2, 6, 6])
